# 🛡️ GRACE Đợt 3: Đối Đầu B3 (Contrastive Demonstration Selection) vs B2 (Baseline)

**Môi Trường Thực Thi**: Kaggle GPU (T4 / P100) với Google Cloud Vertex AI MaaS  
**Hợp Đồng Đánh Giá**: `evaluation_contract.yaml`  
**Kế Hoạch Khoa Học**: `project_docs/plans/PLAN_SUPERVISOR_REQUIREMENTS_EXECUTION_20261014.md`  

### 1. Mục Tiêu Khoa Học:
- So sánh đối đầu trực diện giữa **Baseline B2** (1-shot hybrid CodeT5 768-d + GraphSim) và **B3** (2-shot Contrastive Demonstration Selection: Security-Aware Retriever + Strategy C Counterexample Pair).
- Kiểm chứng giả thuyết: Cặp mẫu tương phản *(1 Vulnerable + 1 Safe)* kết hợp phân tích điểm khác biệt an ninh cốt lõi (*Security Difference*) giúp:
  1. Nâng cao vượt trội khả năng phát hiện lỗ hổng thực tế (**Recall**, **F1-Score**, **MCC**, **Balanced Accuracy**).
  2. Triệt tiêu hoàn toàn sự phụ thuộc/học vẹt nhãn đảo (**Label Sensitivity**) vốn gây lệch 31.0% ở B2 (do 95% mẫu demo B2 bị rơi vào nhãn Safe).
- Đo lường thống kê mẫu ghép cặp: **Kiểm định McNemar** và **Paired Bootstrap 95% Confidence Interval** (2,000 resamples).

### 2. Cấu Hình Thực Nghiệm B3 Chuẩn Figure 6 & Hợp Đồng:
- **Target Query**: Code mục tiêu ($\le 3000$ ký tự) + Đồ thị CPG Joern (tối đa 25 nodes, 25 edges).
- **1 Vulnerable Demo Reference**: Code mẫu có lỗ hổng ($\le 1200$ ký tự) + Nhãn `Vulnerable (1)`.
- **1 Safe Demo Reference**: Code mẫu an toàn tương phản ($\le 1200$ ký tự) + Nhãn `Non-vulnerable (0)`.
- **Core Security Difference**: Phân tích ranh giới an ninh cốt lõi (sanitization guards, bounds check, pointer validation).
- **Backbone LLM**: `publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas` (Location: `us-south1`, Temperature: 0.0, Seed: 42, Max tokens: 256, Retry: 5 attempts).

## Bước 1: Thiết Lập Thư Mục Làm Việc (Workspace Setup)
Tìm kiếm mã nguồn dự án trong `/kaggle/input` hoặc sao chép sang `/kaggle/working/GRACE` để có toàn quyền ghi file báo cáo và checkpoints.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

dest_dir = '/kaggle/working/GRACE'

# 1. Đảm bảo luôn lấy mã nguồn mới nhất từ GitHub
if not os.path.exists(dest_dir):
    print('[*] Đang clone mã nguồn GRACE mới nhất từ GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/HuuHieu03/GRACE.git', dest_dir], check=True)
else:
    print('[*] Đang cập nhật mã nguồn mới nhất từ GitHub...')
    subprocess.run(['git', '-C', dest_dir, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', dest_dir, 'reset', '--hard', 'origin/main'], check=True)

subprocess.run(['git', '-C', dest_dir, 'log', '-1', '--oneline'], check=True)

# 2. Chuyển thư mục làm việc và bổ sung sys.path
%cd /kaggle/working/GRACE
for p in ['/kaggle/working/GRACE', '/kaggle/working/GRACE/src', '/kaggle/working/GRACE/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✓ Thư mục làm việc hiện tại: {os.getcwd()}')
!ls -la


## Bước 2: Cài Đặt Các Gói Phụ Thuộc (Dependencies Installation)
Cài đặt các thư viện cần thiết: `google-auth`, `openai>=1.0.0`, `transformers`, `torch`, `scikit-learn`, `datasets`, `pyyaml`, `pandas`, `tqdm`.

In [ ]:
!pip install -q google-auth google-auth-oauthlib openai>=1.0.0 transformers torch scikit-learn datasets pandas tqdm pyyaml pytest
import torch
print(f'✓ PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✓ GPU Device: {torch.cuda.get_device_name(0)}')
print('✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng!')

## Bước 3: Xác Thực Google Cloud Vertex AI Qua Kaggle Secrets
Đọc Service Account Key từ Kaggle Secret `GCP_SA_KEY`, ghi ra file tạm `/tmp/gcp_sa_key.json`,
thiết lập `GOOGLE_APPLICATION_CREDENTIALS` và kiểm tra sinh Access Token hợp lệ qua `google.auth`.

In [ ]:
import os
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Đọc Kaggle Secret: GCP_SA_KEY
print('[*] Đang nạp Service Account Key từ Kaggle Secrets...')
try:
    user_secrets = UserSecretsClient()
    sa_json_str = user_secrets.get_secret('GCP_SA_KEY')
    sa_path = Path('/tmp/gcp_sa_key.json')
    with open(sa_path, 'w', encoding='utf-8') as f:
        f.write(sa_json_str.strip())
    os.chmod(sa_path, 0o600)
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(sa_path)
    print(f'✓ [OK] Đã lưu Service Account key tại: {sa_path}')
except Exception as e:
    print(f'[!] Cảnh báo nạp Kaggle Secret: {e}. Vui lòng kiểm tra Add-ons -> Secrets trên Kaggle!')

# 2. Khởi tạo cấu hình và xác thực Vertex AI
from config import config, setup_vertex_credentials
cred_file = setup_vertex_credentials()
print(f'[*] File Credentials hoạt động: {cred_file}')
print(f'[*] Vertex Project ID: {config.vertex_project_id}')
print(f'[*] Vertex Location:   {config.vertex_location}')
print(f'[*] Vertex Model:      {config.vertex_model}')

# 3. Xác thực sinh OAuth2 Token qua google.auth
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    creds.refresh(Request())
    print(f'✓ [OK] Vertex AI Authentication Verified! Token prefix: {creds.token[:12]}... (valid={creds.valid})')
except Exception as e:
    print(f'[!] Lỗi xác thực Vertex AI: {e}')

## Bước 4: Kiểm Tra Tập Dữ Liệu & Rào Chắn Hợp Đồng
Xác nhận tập kiểm thử `data/splits/devign_dev_clean_100_ids.json` và kho huấn luyện `data/processed/devign_train_compact_cpg.json.gz` (21,854 mẫu):
- Đạt chuẩn 100 mẫu sạch không rò rỉ mã.
- 100% mẫu kiểm thử có `len(nodes) > 0` và `len(edges) > 0` từ Joern 0.2.5 CPG.

In [ ]:
from pathlib import Path
import json

split_path = Path('data/splits/devign_dev_clean_100_ids.json')
assert split_path.exists(), f'Không tìm thấy file split: {split_path}'

with open(split_path, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print(f'[*] Dataset: {meta.get("dataset")} | Split: {meta.get("split_name")}')
print(f'[*] Tổng số mẫu: {meta.get("total_samples")} (Vuln: {meta.get("vulnerable_count")}, Safe: {meta.get("safe_count")})')
print(f'[*] Seed: {meta.get("seed")} | Hash SHA-256: {meta.get("split_id_hash_sha256_short")}')

sample_details = meta.get("sample_details", [])
assert len(sample_details) == 100, f'Số mẫu chi tiết phải là 100, nhận được: {len(sample_details)}'
for idx, s in enumerate(sample_details):
    nodes = s.get("nodes", [])
    edges = s.get("edges", [])
    assert len(nodes) > 0, f'LỖI: Mẫu {s.get("id")} thiếu CPG nodes!'
    assert len(edges) > 0, f'LỖI: Mẫu {s.get("id")} thiếu CPG edges!'

print(f'✓ [OK] RÀO CHẮN HỢP ĐỒNG ĐẠT CHUẨN: 100% ({len(sample_details)}/100) mẫu có đầy đủ CPG nodes và edges.')


## Bước 5: Chạy Unit Tests Xác Minh Hệ Thống Đợt 3
Xác nhận các chốt chặn kỹ thuật: Cấu trúc prompt contrastive 2-shot, độ dài cắt ngắn, kiểm định McNemar và Paired Bootstrap.

In [ ]:
!python -m pytest tests/test_stage3_b3_vs_b2.py tests/test_stage2_contract_compliance.py tests/test_stage5_contrastive_selector.py -v
print('✓ [All Unit Tests Passed] Toàn bộ các chốt chặn Đợt 3 đã hoàn tất xuất sắc!')

## Bước 6: Thực Thi Thí Nghiệm Đợt 3 (Chạy Đối Đầu B3 vs B2)
Thực thi script `scripts/run_stage3_b3_vs_b2.py` với mô hình Vertex AI:
- Model: `publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas`
- Chiến lược chọn cặp: `counterexample_pair` (Strategy C)
- Kho ứng viên: Top-50 ứng viên bằng CodeT5 L2 search từ 21,854 mẫu Train
- Báo cáo lưu tại: `output/stage3/b3_vs_b2_comparison_report.json`

In [ ]:
!python scripts/run_stage3_b3_vs_b2.py \
    --dev_split data/splits/devign_dev_clean_100_ids.json \
    --dataset devign \
    --model publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas \
    --provider "Vertex AI" \
    --location us-south1 \
    --strategy counterexample_pair \
    --lambda_weight 0.3 \
    --output output/stage3/b3_vs_b2_comparison_report.json

# Sao chép kết quả về thư mục gốc /kaggle/working để Kaggle nén xuất đầy đủ
!mkdir -p /kaggle/working/output/stage3
!cp output/stage3/b3_vs_b2_comparison_report.json /kaggle/working/output/stage3/ 2>/dev/null || true
!cp output/stage3/b3_vs_b2_comparison_report.csv /kaggle/working/output/stage3/ 2>/dev/null || true
!cp output/stage3/b3_vs_b2_comparison_report.json /kaggle/working/b3_vs_b2_comparison_report.json 2>/dev/null || true
!cp output/stage3/b3_vs_b2_comparison_report.csv /kaggle/working/b3_vs_b2_comparison_report.csv 2>/dev/null || true


## Bước 7: Phân Tích & Nghiệm Thu Kết QuẢ Thực Nghiệm Đợt 3
Đọc file báo cáo JSON, phân tích chi tiết mức độ cải thiện của B3 so với B2 trên tất cả các chỉ số:
F1-Score, MCC, Balanced Accuracy, Precision, Recall, McNemar p-value, Bootstrap 95% CI và Độ nhạy nhãn đảo.

In [ ]:
import json
from pathlib import Path

rep_candidates = [
    Path('output/stage3/b3_vs_b2_comparison_report.json'),
    Path('/kaggle/working/output/stage3/b3_vs_b2_comparison_report.json'),
    Path('/kaggle/working/b3_vs_b2_comparison_report.json')
]

report_file = None
for p in rep_candidates:
    if p.exists() and p.stat().st_size > 100:
        report_file = p
        break

if report_file:
    with open(report_file, 'r', encoding='utf-8') as f:
        rep = json.load(f)
    
    b2 = rep.get('b2_baseline', {})
    b3 = rep.get('b3_contrastive', {})
    diff = rep.get('head_to_head_delta', {})
    ls = rep.get('label_sensitivity_comparison', {})
    mcnemar = rep.get('mcnemar_test', {})
    boot = rep.get('paired_bootstrap_95_ci', {})
    
    print('=' * 88)
    print('  TỔNG HỢP SO SÁNH ĐỐI ĐẦU B3 VS B2 TRÊN VERTEX AI (QWEN-3 CODER 480B)')
    print('=' * 88)
    print(f'[*] Mẫu hợp lệ: {rep.get("metadata", {}).get("valid_samples")} / {rep.get("metadata", {}).get("total_samples")}')
    print('-' * 88)
    print(f'{"Chỉ số":<22} | {"B2 (Baseline 1-shot)":<20} | {"B3 (Contrastive 2-shot)":<22} | {"Delta (B3 - B2)":<16}')
    print('-' * 88)
    for k in ['accuracy', 'precision', 'recall', 'f1_score', 'mcc', 'balanced_accuracy', 'predicted_vulnerable_rate']:
        vb2 = b2.get(k, 0.0)
        vb3 = b3.get(k, 0.0)
        d = diff.get(f'delta_{k}', vb3 - vb2)
        print(f'{k:<22} | {vb2:<20.4f} | {vb3:<22.4f} | {d:+15.4f}')
    print('-' * 88)
    print(f'[*] ĐỘ NHẠY NHÃN ĐẢO (LABEL SENSITIVITY):')
    print(f'    - B2 Net Label Sensitivity:        {ls.get("b2_net_label_sensitivity_pct")}% (Bị lệch 31.0% do demo Safe)')
    print(f'    - B3 Order/Permutation Sens.:      {ls.get("b3_order_sensitivity_pct")}%')
    print(f'    - Triệt tiêu thiên lệch (Reduct.):  {ls.get("sensitivity_reduction_pct")}%')
    print('-' * 88)
    if mcnemar:
        print(f'[*] McNemar Test: Chi2={mcnemar.get("chi2_statistic")}, p-value={mcnemar.get("p_value")} (Significant at p<0.05: {mcnemar.get("is_significant_p05")})')
    if boot:
        print(f'[*] Bootstrap 95% CI Delta F1:  {boot.get("delta_f1", {}).get("ci_95")} (Excludes 0: {boot.get("delta_f1", {}).get("excludes_zero")})')
        print(f'[*] Bootstrap 95% CI Delta MCC: {boot.get("delta_mcc", {}).get("ci_95")} (Excludes 0: {boot.get("delta_mcc", {}).get("excludes_zero")})')
    print('=' * 88)
else:
    print('[!] Chưa tìm thấy file báo cáo output/stage3/b3_vs_b2_comparison_report.json.')